# Ubicación geográfica de las estaciones
Explorando  [datos abiertos de Ecobici](https://ecobici.cdmx.gob.mx/datos-abiertos/?utm_source=chatgpt.com)

En este código se mapea cada una de las estaciones de Ecobici con la alcaldía correspondiente.

In [1]:
import requests
import json
import pandas as pd
# 1. Abrir el JSON principal
url = "https://gbfs.mex.lyftbikes.com/gbfs/gbfs.json"
response = requests.get(url)
data = response.json()
# 2. Buscar la URL de station_information
feeds = data["data"]["en"]["feeds"]

station_info_url = [
    feed["url"]
    for feed in feeds
    if feed["name"] == "station_information"
][0]

print("URL de station_information:")
print(station_info_url)

# 3. Abrir station_information
response_station = requests.get(station_info_url)
station_data = response_station.json()

# 4. Ver TODO el JSON
print(json.dumps(station_data, indent=2, ensure_ascii=False))

Se truncaron las últimas líneas 5000 del resultado de transmisión.
        "station_id": "395",
        "external_id": "754358a9-f978-4f21-bd42-62372b8f120a",
        "name": "CE-296 Diagonal San Antonio-Enrique Rébsamen",
        "short_name": "296",
        "lat": 19.394283,
        "lon": -99.158756,
        "rental_methods": [
          "CREDITCARD",
          "KEY"
        ],
        "capacity": 23,
        "electric_bike_surcharge_waiver": false,
        "is_charging": false,
        "eightd_has_key_dispenser": false,
        "has_kiosk": true
      },
      {
        "station_id": "396",
        "external_id": "df7943fd-044b-444f-a194-c0de2ff4c3ba",
        "name": "CE-323 Concepción Beistegui-Gabriel Mancera",
        "short_name": "323",
        "lat": 19.388986312246836,
        "lon": -99.1632449628014,
        "rental_methods": [
          "CREDITCARD",
          "KEY"
        ],
        "capacity": 27,
        "electric_bike_surcharge_waiver": false,
        "is_charging":

Buscamos identificar cada estación (señalada con 'station_id') a partir de las coordenadas que tiene en los datos abiertos y se cruzan con los datos del polígono que contiene cada alacaldía.


In [2]:
import shutil
import os
from google.colab import drive

# 1. Limpiar cualquier carpeta local falsa llamada /content/drive
shutil.rmtree("/content/drive", ignore_errors=True)

# 2. Montar Google Drive correctamente
drive.mount("/content/drive")

# 3. Verificar que sí estás viendo tu Drive real
ruta = "/content/drive/MyDrive/Tesis_caso"


Mounted at /content/drive


In [3]:
# -*- coding: utf-8 -*-
"""
Identifica en qué alcaldía cae cada estación de Ecobici.

Combina dos fuentes que NO vienen juntas en ningún dataset oficial:
  1. Ubicación de estaciones (lat/lon) -> feed oficial GBFS de Ecobici
     https://gbfs.mex.lyftbikes.com/gbfs/gbfs.json
     (sistema actual operado con Lyft; coincide con los CSVs de viajes
     2024-2025 que ya estás descargando).
  2. Polígonos de las 16 alcaldías -> Marco Geoestadístico INEGI 2020,
     republicado como archivo estático (GitHub Pages) porque
     datos.cdmx.gob.mx descarta las conexiones desde IPs de nube (timeout).

Hace un cruce espacial (point-in-polygon): cada estación se ubica dentro
del polígono de alcaldía que la contiene.

Uso en Colab:
    1. Ejecuta esta celda completa (instala geopandas si falta).
    2. Autoriza el montaje de Drive si no está montado ya.
    3. Resultado en:
       /content/drive/MyDrive/Tesis_caso/catalogo_ecobici_estaciones_alcaldia.csv
"""

import os
import json
import certifi
import requests

# ------------------------------------------------------------------
# Dependencias geoespaciales (geopandas trae shapely)
# ------------------------------------------------------------------
try:
    import geopandas as gpd
    from shapely.geometry import Point
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas"], check=True)
    import geopandas as gpd
    from shapely.geometry import Point

# ------------------------------------------------------------------
# Configuración
# ------------------------------------------------------------------
GBFS_ROOT = "https://gbfs.mex.lyftbikes.com/gbfs/gbfs.json"
# Polígonos de alcaldías: misma geometría oficial (Marco Geoestadístico INEGI
# 2020) republicada como archivo estático en GitHub Pages. Se usa esta fuente
# directamente porque datos.cdmx.gob.mx descarta las conexiones desde IPs de
# nube/datacenter como las de Colab (timeout, no error de certificado).
ALCALDIAS_FALLBACK_URL = "https://mpjashby.github.io/crimemappingdata/cdmx_alcaldias.gpkg"
DRIVE_ROOT = "/content/drive/MyDrive/Tesis_caso"
OUT_CSV = os.path.join(DRIVE_ROOT, "catalogo_ecobici_estaciones_alcaldia.csv")
OUT_CSV_RESUMEN = os.path.join(DRIVE_ROOT, "ecobici_estaciones_por_alcaldia_resumen.csv")

os.makedirs(DRIVE_ROOT, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; ColabDataCollector/1.0)"})
session.verify = certifi.where()


# ------------------------------------------------------------------
# 1) Estaciones de Ecobici vía GBFS (feed oficial en tiempo real)
# ------------------------------------------------------------------
def get_ecobici_stations():
    root = session.get(GBFS_ROOT, timeout=30).json()

    # El feed raíz lista los sub-feeds disponibles por idioma
    feeds = []
    for lang_block in root["data"].values():
        feeds.extend(lang_block["feeds"])

    info_url = next(f["url"] for f in feeds if f["name"] == "station_information")
    info = session.get(info_url, timeout=30).json()
    stations = info["data"]["stations"]

    rows = []
    for s in stations:
        rows.append({
            "station_id": s.get("station_id"),
            "name": s.get("name"),
            "short_name": s.get("short_name"),
            "lat": s.get("lat"),
            "lon": s.get("lon"),
        })
    return rows


# ------------------------------------------------------------------
# 2) Polígonos de alcaldías vía fuente alterna (GitHub Pages)
# ------------------------------------------------------------------
def get_alcaldias_from_fallback():
    resp = session.get(ALCALDIAS_FALLBACK_URL, timeout=30)
    resp.raise_for_status()
    gpkg_path = "/content/alcaldias.gpkg"
    with open(gpkg_path, "wb") as f:
        f.write(resp.content)
    return gpd.read_file(gpkg_path)


def get_alcaldias_geodataframe():
    print("  Descargando polígonos de alcaldías (fuente alterna, GitHub Pages)...")
    gdf = get_alcaldias_from_fallback()
    print("  ✔ Obtenido")

    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    else:
        gdf = gdf.to_crs(epsg=4326)
    return gdf


def find_alcaldia_field(gdf):
    """Detecta el nombre de columna que contiene el nombre de la alcaldía."""
    candidatos = ["NOMGEO", "nomgeo", "NOMBRE", "nombre", "alcaldia",
                  "ALCALDIA", "nom_mun", "NOM_MUN"]
    for c in candidatos:
        if c in gdf.columns:
            return c
    # Si ninguno calza, mostramos las columnas para que el usuario decida
    raise RuntimeError(
        "No se identificó automáticamente la columna con el nombre de la "
        f"alcaldía. Columnas disponibles: {list(gdf.columns)}. "
        "Ajusta 'find_alcaldia_field' con el nombre correcto."
    )


# ------------------------------------------------------------------
# 3) Cruce espacial (point-in-polygon)
# ------------------------------------------------------------------
def main():
    print("Descargando estaciones de Ecobici (GBFS)...")
    stations = get_ecobici_stations()
    print(f"  {len(stations)} estaciones obtenidas.")

    print("Descargando polígonos de alcaldías...")
    alcaldias_gdf = get_alcaldias_geodataframe()
    campo_alcaldia = find_alcaldia_field(alcaldias_gdf)
    print(f"  {len(alcaldias_gdf)} alcaldías cargadas (columna: '{campo_alcaldia}').")

    print("Haciendo cruce espacial (point-in-polygon)...")
    stations_gdf = gpd.GeoDataFrame(
        stations,
        geometry=[Point(s["lon"], s["lat"]) for s in stations],
        crs="EPSG:4326",
    )

    joined = gpd.sjoin(
        stations_gdf,
        alcaldias_gdf[[campo_alcaldia, "geometry"]],
        how="left",
        predicate="within",
    )
    joined = joined.rename(columns={campo_alcaldia: "alcaldia"})

    sin_alcaldia = joined["alcaldia"].isna().sum()
    if sin_alcaldia:
        print(f"  ⚠️  {sin_alcaldia} estaciones no cayeron dentro de ningún "
              "polígono (posibles coordenadas fuera de CDMX o en el borde).")

    resultado = joined[["station_id", "name", "short_name", "alcaldia"]]
    resultado = resultado.sort_values(["alcaldia", "name"])
    resultado.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    print(f"\n✅ Guardado: {OUT_CSV}")

    # --- Resumen: cantidad de estaciones por alcaldía ---
    resumen_df = (
        resultado["alcaldia"]
        .value_counts(dropna=False)
        .rename_axis("alcaldia")
        .reset_index(name="num_estaciones")
        .sort_values("num_estaciones", ascending=False)
        .reset_index(drop=True)
    )
    resumen_df.to_csv(OUT_CSV_RESUMEN, index=False, encoding="utf-8-sig")

    print(f"✅ Guardado resumen: {OUT_CSV_RESUMEN}")
    print("\nEstaciones por alcaldía:")
    print(resumen_df.to_string(index=False))

    return resultado, resumen_df


if __name__ == "__main__":
    estaciones_alcaldia_df, estaciones_por_alcaldia_df = main()

Descargando estaciones de Ecobici (GBFS)...
  677 estaciones obtenidas.
Descargando polígonos de alcaldías...
  Descargando polígonos de alcaldías (fuente alterna, GitHub Pages)...
  ✔ Obtenido
  16 alcaldías cargadas (columna: 'nomgeo').
Haciendo cruce espacial (point-in-polygon)...

✅ Guardado: /content/drive/MyDrive/Tesis_caso/catalogo_ecobici_estaciones_alcaldia.csv
✅ Guardado resumen: /content/drive/MyDrive/Tesis_caso/ecobici_estaciones_por_alcaldia_resumen.csv

Estaciones por alcaldía:
      alcaldia  num_estaciones
    Cuauhtémoc             243
 Benito Juárez             213
Miguel Hidalgo             139
      Coyoacán              37
  Azcapotzalco              28
Álvaro Obregón              17


Estaciones por alcaldía:
alcaldia
Cuauhtémoc        243
Benito Juárez     213
Miguel Hidalgo    139
Coyoacán           37
Azcapotzalco       28
Álvaro Obregón     17

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
#Necesito explorar los datos de la Alcaldía ya que hacen falta algunas alcaldías
import pandas as pd

# Cambia esta ruta por la de tu archivo
RUTA_ARCHIVO = "/content/drive/MyDrive/Tesis_caso/catalogo_ecobici_estaciones_alcaldia.csv"

# Leer archivo
df = pd.read_csv(RUTA_ARCHIVO)

# Ver dimensiones
print("Dimensiones:", df.shape)

# Mostrar columnas
print("\nColumnas:")
for col in df.columns:
    print("-", col)

# Ver primeras filas
display(df.head())

Dimensiones: (677, 4)

Columnas:
- station_id
- name
- short_name
- alcaldia


,station_id,name,short_name,alcaldia
0,673,CE-508 Libertad-C. José Sánchez Trujillo,508,Azcapotzalco
1,681,CE-509 Juárez - Venecia,509,Azcapotzalco
2,575,CE-510 Egipto - Calle Norte 79 A,510,Azcapotzalco
3,697,CE-511 Palestina - Egipto,511,Azcapotzalco
4,583,CE-512 Clavería - Floresta,512,Azcapotzalco


In [6]:
df_ordenado = df.sort_values("station_id").reset_index(drop=True)

display(df_ordenado.head(20))

,station_id,name,short_name,alcaldia
0,1,CE-710 Molino del Rey - Glorieta de la Lealtad,710,Miguel Hidalgo
1,5,CE-407 Prolongación Xochicalco-General Emilia...,407,Benito Juárez
2,6,CE-428 Prolongación Uxmal-Av. Popocatépetl (E...,428,Benito Juárez
3,7,CE-483 Colegio Salesiano - Marina Nacional,483,Miguel Hidalgo
4,8,CE-443 Bruno Traven-Golondrinas,443,Benito Juárez
5,9,CE-423 Moras-José María Rico (Eje 8),423,Benito Juárez
6,10,CE-438 Adolfo Prieto-José María Olloqui,438,Benito Juárez
7,11,CE-415 San Francisco-Parroquia,415,Benito Juárez
8,12,CE-417 Goya-Augusto Rodin,417,Benito Juárez
9,14,CE-022 Reforma - Manchester,022,Cuauhtémoc
